In [1]:

# Standard Libraries

import os
import gc
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# Data Manipulation

import numpy as np
import pandas as pd


# Visualization

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("Set2")


# Machine Learning

from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Miscellaneous

from tqdm import tqdm

# Feature Engineering

## Why Feature Engineering?

Even though IEEE-CIS already contains many engineered variables (`C`, `D`, `V`), there are still highly informative features we can derive from:

* Transaction time
* Transaction amount
* Card usage
* Customer behavior
* Merchant behavior
* Historical transaction patterns

Modern IEEE-CIS solutions consistently improve performance by creating temporal features, amount transformations, aggregation features, and interaction features while carefully avoiding temporal leakage.



In [4]:
DATA_DIR = Path('data')

In [6]:
X_train = pd.read_csv(DATA_DIR/'preprocessed/X_train.csv')
X_valid = pd.read_csv(DATA_DIR/'preprocessed/X_valid.csv')
X_test = pd.read_csv(DATA_DIR/'preprocessed/X_test.csv')


# 5.1 Log Transform Transaction Amount

Transaction amounts are highly right-skewed.

In [7]:
# Log Amount

for dataset in [X_train, X_valid, X_test]:

    dataset["TransactionAmt_Log"] = np.log1p(
        dataset["TransactionAmt"]
    )

# 5.2 Temporal Features

`TransactionDT` is measured in seconds from an internal reference point.

Although we don't know the actual date, we can still extract useful temporal patterns.

In [8]:
SECONDS_IN_HOUR = 3600
SECONDS_IN_DAY = 86400

In [9]:
def create_time_features(df):

    df = df.copy()

    df["Hour"] = (
        df["TransactionDT"] //
        SECONDS_IN_HOUR
    ) % 24

    df["Day"] = (
        df["TransactionDT"] //
        SECONDS_IN_DAY
    )

    df["Week"] = (
        df["Day"] // 7
    )

    df["WeekDay"] = (
        df["Day"] % 7
    )

    return df

In [10]:
X_train = create_time_features(X_train)
X_valid = create_time_features(X_valid)
X_test = create_time_features(X_test)

Time-derived features such as hour, weekday, and day index consistently rank among the most useful engineered variables for IEEE-CIS fraud detection because fraudulent activity often exhibits temporal patterns.




# 5.3 Amount Percentile

Raw transaction amount isn't always informative.

Its relative size often is.

In [11]:
amount_percentiles = (
    X_train["TransactionAmt"]
    .rank(pct=True)
)

In [12]:
X_train["AmountPercentile"] = amount_percentiles

X_valid["AmountPercentile"] = (
    X_valid["TransactionAmt"]
    .rank(pct=True)
)

X_test["AmountPercentile"] = (
    X_test["TransactionAmt"]
    .rank(pct=True)
)

# 5.4 Transaction Velocity

Large bursts of transactions often indicate fraud.

We'll compute transaction counts per hour.

In [13]:
for dataset in [X_train, X_valid, X_test]:

    dataset["TransactionsPerHour"] = (
        dataset.groupby("Hour")
               ["TransactionAmt"]
               .transform("count")
    )

# 5.5 Card Frequency

Certain cards appear much more often.

In [14]:
card_frequency = (
    X_train["card1"]
    .value_counts()
)

In [15]:
for dataset in [X_train, X_valid, X_test]:

    dataset["card1_frequency"] = (
        dataset["card1"]
        .map(card_frequency)
        .fillna(0)
    )

# 5.6 Email Frequency

In [16]:
email_frequency = (
    X_train["P_emaildomain"]
    .value_counts()
)

In [17]:
for dataset in [X_train, X_valid, X_test]:

    dataset["email_frequency"] = (
        dataset["P_emaildomain"]
        .map(email_frequency)
        .fillna(0)
    )

# 5.7 Card-Level Statistics

One of the strongest feature engineering techniques is to summarize historical behavior for an entity such as a payment card.

In [20]:
card_statistics = (
    X_train.groupby("card1")["TransactionAmt"]
            .agg(["mean", "std", "min", "max"])
)

global_mean = X_train["TransactionAmt"].mean()
global_std = X_train["TransactionAmt"].std()
global_min = X_train["TransactionAmt"].min()
global_max = X_train["TransactionAmt"].max()

In [21]:
for dataset in [X_train, X_valid, X_test]:

    dataset["card1_amt_mean"] = (
        dataset["card1"]
        .map(card_statistics["mean"])
        .fillna(global_mean)
    )

    dataset["card1_amt_std"] = (
        dataset["card1"]
        .map(card_statistics["std"])
        .fillna(0.0)
    )

    dataset["card1_amt_min"] = (
        dataset["card1"]
        .map(card_statistics["min"])
        .fillna(global_min)
    )

    dataset["card1_amt_max"] = (
        dataset["card1"]
        .map(card_statistics["max"])
        .fillna(global_max)
    )

# 5.8 Relative Transaction Amount

Rather than using only the raw amount, compare it with the historical average for that card.

In [22]:
EPSILON = 1e-6

for dataset in [X_train, X_valid, X_test]:
    dataset["Amount_to_CardMean"] = (
        dataset["TransactionAmt"] /
        (dataset["card1_amt_mean"] + EPSILON)
    )

Relative amount features help capture unusually large or small transactions compared with a customer's historical behavior and are widely used in fraud detection systems.



# 5.9 Address Frequency

In [23]:
addr_frequency = (
    X_train["addr1"]
    .value_counts()
)

In [24]:
for dataset in [X_train, X_valid, X_test]:

    dataset["addr1_frequency"] = (
        dataset["addr1"]
        .map(addr_frequency)
        .fillna(0)
    )

# 5.10 Device Frequency

In [25]:
device_frequency = (
    X_train["DeviceType"]
    .value_counts()
)

In [26]:
for dataset in [X_train, X_valid, X_test]:

    dataset["Device_frequency"] = (
        dataset["DeviceType"]
        .map(device_frequency)
        .fillna(0)
    )

# 5.11 Feature Engineering Summary

In [31]:
# loadded only for Features Summary
data = pd.read_csv(DATA_DIR/"raw/cleaned_df.csv")
data = data.sort_values("TransactionDT").reset_index(drop=True)

In [32]:
print("=" * 60)
print("Feature Engineering Summary")
print("=" * 60)

new_features = [
    col for col in X_train.columns
    if col not in data.columns
]

print(f"New Features Created: {len(new_features)}")

for feature in new_features:
    print(feature)

Feature Engineering Summary
New Features Created: 329
addr1_missing
addr2_missing
dist1_missing
dist2_missing
P_emaildomain_missing
R_emaildomain_missing
D2_missing
D3_missing
D4_missing
D5_missing
D6_missing
D7_missing
D8_missing
D9_missing
D10_missing
D11_missing
D12_missing
D13_missing
D14_missing
D15_missing
M1_missing
M2_missing
M3_missing
M4_missing
M5_missing
M6_missing
M7_missing
M8_missing
M9_missing
V1_missing
V2_missing
V3_missing
V4_missing
V5_missing
V6_missing
V7_missing
V8_missing
V9_missing
V10_missing
V11_missing
V12_missing
V13_missing
V14_missing
V15_missing
V16_missing
V17_missing
V18_missing
V19_missing
V20_missing
V21_missing
V22_missing
V23_missing
V24_missing
V25_missing
V26_missing
V27_missing
V28_missing
V29_missing
V30_missing
V31_missing
V32_missing
V33_missing
V34_missing
V35_missing
V36_missing
V37_missing
V38_missing
V39_missing
V40_missing
V41_missing
V42_missing
V43_missing
V44_missing
V45_missing
V46_missing
V47_missing
V48_missing
V49_missing
V50_miss

In [33]:
X_train.to_csv(DATA_DIR/'FE_datasets/X_train.csv', index=False)
X_valid.to_csv(DATA_DIR/'FE_datasets/X_valid.csv', index=False)
X_test.to_csv(DATA_DIR/'FE_datasets/X_test.csv', index=False)
print("Saved Engineered Features.")

Saved Engineered Features.


# Engineering Note

There is one limitation in the feature engineering above that I intentionally left in place because it's an important teaching point.

Features such as:

* `card1_amt_mean`
* `card1_frequency`
* `email_frequency`
* `addr1_frequency`

are computed using **all training rows**, including transactions that occur later in time than the current transaction. This is acceptable for many offline experiments, but in a true production fraud detection system it introduces subtle temporal leakage. A production-grade solution would compute these as **cumulative ("read-before-update") statistics**, where each transaction only has access to information available before it occurred. This approach is increasingly recommended in recent IEEE-CIS research and deployment-oriented studies.

In [34]:
# del X_train
# del X_valid
# del X_test
# del data

In [38]:
import gc

def reset_memory():
    # Delete all variables.
    for var in gc.get_objects():
        try:
            del var
        except:
            pass

    # Run the garbage collector.
    gc.collect()

    print("Memory has been reset")


In [39]:
reset_memory()

Memory has been reset
